# 🛡️ Zero-Trust Multi-Agent Trading Desk: Capstone Replication Notebook

Welcome to the replication notebook for the **Zero-Trust Multi-Agent Trading Desk**—submitted to the Kaggle *Agents for Business* capstone track.

## ⚡ Quick Pitch
AI agents are vulnerable to prompt injections and hallucinations. Giving them direct access to broker API keys (**Ambient Authority**) is a severe security risk. This system treats the AI swarm as completely untrusted. It routes all proposed trades through a deterministic, math-first Python **Policy Server** and a **Human-in-the-Loop Queue**. If the AI is compromised or goes rogue, the system fails closed.

## 📐 Core Architecture & Invariants
1. **No Agent Credentials:** No agent container holds broker keys.
2. **Pure Python Policy Server:** Safety rules are enforced programmatically without LLMs.
3. **Parallel Consensus:** Fundamental and Technical agents run concurrently; any signal conflict aborts the session.
4. **Data Provenance:** Every market payload includes a SHA256 hash. The Policy Server recomputes it to block data fabrication.
5. **State Amnesia:** Session RAM is immediately purged on termination.
6. **PII Masking:** PII is scrubbed and masked at the ingest boundary.

---
## 📋 1. Inspect the Root Policy Configuration
This YAML file acts as the single source of truth for runtime validation limits. It is read by the Policy Server, never by the agents.

In [ ]:
import yaml
with open('config/policy_config.yaml') as f:
    config = yaml.safe_load(f)

print(yaml.dump(config, default_flow_style=False))

---
## 🧪 2. Run the Full Test Suite
We run the 50 unit and integration tests (including the 10 Golden Dataset test cases) to prove the safety and compliance of the orchestrator.

In [ ]:
!PYTHONPATH=. pytest

---
## 🛡️ 3. Run the Prompt Injection Simulator
This script generates novel prompt injection payloads (Red Team) and tests if the Ingest Filter (Blue Team) successfully intercepts them.

In [ ]:
!PYTHONPATH=. python -m tests.simulate_injection --verbose

---
## 📊 4. Inspect the Audit Timeline
Here we load the append-only telemetry audit trails (`logs/audit.jsonl`) into a pandas DataFrame to visualize decision codes and latency metrics.

In [ ]:
import pandas as pd
import json
import os

audit_file = 'logs/audit.jsonl'
if os.path.exists(audit_file):
    records = []
    with open(audit_file) as f:
        for line in f:
            records.append(json.loads(line))
    df = pd.DataFrame(records)
    print(df[['_written_at', 'decision_code', 'ticker', 'action', 'estimated_value_usd']])
else:
    print('No audit log found. Run tests or simulation cells first to generate logs.')